In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split, WeightedRandomSampler
from torchvision.transforms import v2
from torchvision import models
import matplotlib.pyplot as plt
from collections import Counter
import time

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Executing on: {device}")

Executing on: cuda


In [2]:
EMOTION_MAP = {0: 'Angry', 1: 'Disgust', 2: 'Fear', 3: 'Happy', 4: 'Sad', 5: 'Surprise', 6: 'Neutral'}

class FERDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        pixels = np.stack(dataframe['pixels'].apply(lambda x: np.fromstring(x, sep=' ', dtype=np.float32)).values)
        self.images = pixels.reshape(-1, 1, 48, 48)
        self.images = (self.images - 127.5) / 127.5
        self.labels = dataframe['emotion'].values
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = torch.tensor(self.images[idx], dtype=torch.float32)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        if self.transform:
            img = self.transform(img)
        return img.repeat(3, 1, 1), label

train_transforms = v2.Compose([
    v2.Resize((224, 224), antialias=True),
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomRotation(degrees=10),
    v2.RandomAffine(degrees=0, translate=(0.1, 0.1)),
])

eval_transforms = v2.Compose([
    v2.Resize((224, 224), antialias=True),
])

In [3]:
train_df = pd.read_csv('train_dataset.csv')

train_size = int(0.8 * len(train_df))
val_size = len(train_df) - train_size
train_split, val_split = random_split(train_df, [train_size, val_size], generator=torch.Generator().manual_seed(RANDOM_SEED))

train_df_concrete = train_df.iloc[train_split.indices].reset_index(drop=True)
val_df_concrete = train_df.iloc[val_split.indices].reset_index(drop=True)

# Class Imbalance
train_labels = train_df_concrete['emotion'].values
class_weights = {cls: 1.0 / count for cls, count in Counter(train_labels).items()}
sample_weights = [class_weights[label] for label in train_labels]
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

train_dataset = FERDataset(train_df_concrete, transform=train_transforms)
val_dataset = FERDataset(val_df_concrete, transform=eval_transforms)

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

In [4]:
# EfficientNet-B0
class EfficientFER(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()
        # Load pre-trained EfficientNet-b0
        weights = models.EfficientNet_B0_Weights.DEFAULT
        self.backbone = models.efficientnet_b0(weights=weights)

        # UNFREEZE the last few layers to allow micro-feature learning
        for name, param in self.backbone.named_parameters():
            if not name.startswith('features.7') and not name.startswith('features.8'):
                param.requires_grad = False

        # Replace the final classification head
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(p=0.4, inplace=True),
            nn.Linear(in_features, num_classes)
        )

    def forward(self, x):
        return self.backbone(x)

model = EfficientFER(num_classes=7).to(device)

print(f"Total Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 50.4MB/s]


Total Parameters: 4,016,515
Trainable Parameters: 1,138,359


In [5]:
# Optimization Loop
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=5e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

EPOCHS = 25

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)

    train_loss /= len(train_loader.dataset)

    # Validation
    model.eval()
    val_loss, corrects = 0.0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            val_loss += criterion(outputs, labels).item() * images.size(0)
            corrects += (outputs.argmax(1) == labels).sum().item()

    val_loss /= len(val_loader.dataset)
    val_acc = corrects / len(val_loader.dataset)

    scheduler.step(val_acc)

    print(f"Epoch {epoch+1:02d}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

torch.save(model.state_dict(), 'efficientnet_fer_sota.pth')

Epoch 01/25 | Train Loss: 1.7062 | Val Loss: 1.6572 | Val Acc: 0.3390
Epoch 02/25 | Train Loss: 1.4208 | Val Loss: 1.4866 | Val Acc: 0.4370
Epoch 03/25 | Train Loss: 1.2667 | Val Loss: 1.4924 | Val Acc: 0.4400
Epoch 04/25 | Train Loss: 1.1750 | Val Loss: 1.4188 | Val Acc: 0.4640
Epoch 05/25 | Train Loss: 1.1098 | Val Loss: 1.3564 | Val Acc: 0.4990
Epoch 06/25 | Train Loss: 1.0793 | Val Loss: 1.4299 | Val Acc: 0.4550
Epoch 07/25 | Train Loss: 1.0111 | Val Loss: 1.3576 | Val Acc: 0.5240
Epoch 08/25 | Train Loss: 0.9611 | Val Loss: 1.3558 | Val Acc: 0.5120
Epoch 09/25 | Train Loss: 0.9348 | Val Loss: 1.3935 | Val Acc: 0.4920
Epoch 10/25 | Train Loss: 0.9137 | Val Loss: 1.3879 | Val Acc: 0.5170
Epoch 11/25 | Train Loss: 0.8621 | Val Loss: 1.3708 | Val Acc: 0.5270
Epoch 12/25 | Train Loss: 0.8340 | Val Loss: 1.3974 | Val Acc: 0.5280
Epoch 13/25 | Train Loss: 0.8275 | Val Loss: 1.3839 | Val Acc: 0.5220
Epoch 14/25 | Train Loss: 0.7797 | Val Loss: 1.4160 | Val Acc: 0.5280
Epoch 15/25 | Train 